# FLUX.2 Pro via OpenRouter

This notebook generates and edits images with **Black Forest Labs FLUX.2 Pro** through the
[OpenRouter API](https://openrouter.ai/docs) — one provider-agnostic endpoint for every image model.

Sections:

- **Text to Image** — FLUX.2 Pro, prompt-driven generation with aspect ratio, resolution, seed, and output format
- **Image Edit** — FLUX.2 Pro, edit an uploaded image via `input_references`
- **Budget** — FLUX.2 Klein 4B, the fastest and cheapest FLUX.2 variant for drafts

You will need an OpenRouter API key: <https://openrouter.ai/settings/keys>.

Have fun and do great things.

In [ ]:
# @title Install requirements
from io import BytesIO
import base64
import IPython
import os
from PIL import Image
import requests

from google.colab import files, output
from IPython.display import display

In [ ]:
'''
Optional:
'''
from google.colab import drive  # Link to your private Google Drive storage

# Request access authorization for your Google Drive storage
drive.mount('./drive')

In [ ]:
# @title Connect to the OpenRouter API

import getpass
from google.colab import userdata

try:
    # Store the key as a Colab secret named OPENROUTER_API_KEY (key icon in the left sidebar)
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key")

In [ ]:
# @title Define functions

OPENROUTER_IMAGES_URL = "https://openrouter.ai/api/v1/images"


def generate_image(
    model,
    prompt,
    aspect_ratio=None,
    resolution=None,
    seed=0,
    output_format="png",
    input_references=None,
    n=1,
):
    """Call the OpenRouter image endpoint and return a list of PIL Images."""
    params = {
        "model": model,
        "prompt": prompt,
        "n": n,
        "output_format": output_format,
    }
    if aspect_ratio:
        params["aspect_ratio"] = aspect_ratio
    if resolution:
        params["resolution"] = resolution
    if seed:
        params["seed"] = seed  # 0 means random
    if input_references:
        params["input_references"] = input_references

    response = requests.post(
        OPENROUTER_IMAGES_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json=params,
    )
    response.raise_for_status()

    return [
        Image.open(BytesIO(base64.b64decode(item["b64_json"])))
        for item in response.json()["data"]
    ]


def to_data_url(path):
    """Encode a local or Drive image file as a base64 data URL."""
    mime = "image/" + (os.path.splitext(path)[1].lstrip(".").lower() or "png")
    with open(path, "rb") as f:
        return "data:" + mime + ";base64," + base64.b64encode(f.read()).decode("utf-8")


def show_and_save(images, output_format="png", prefix="flux"):
    """Display each generated image and save it next to the notebook."""
    for i, image in enumerate(images):
        display(image)
        filename = f"{prefix}_{i}.{output_format}"
        image.save(filename, format=output_format.upper())
        print(f"Saved: {filename}")

# Generate

FLUX.2 Pro is Black Forest Labs' high-end image generation model: strong prompt adherence,
stable lighting, sharp textures, and legible text rendering, up to 4 MP resolution.

See <https://openrouter.ai/black-forest-labs/flux.2-pro>

In [ ]:
# @title FLUX.2 Pro — Text to Image

prompt = "A retro travel poster of the Black Forest at dusk, with the bold legible headline 'FLUX.2 PRO' in art-deco lettering, warm orange sky against deep green treetops, subtle paper texture"  # @param {type:"string"}
aspect_ratio = "1:1"  # @param ["1:1", "16:9", "9:16", "4:3", "3:2", "21:9"]
resolution = "1K"  # @param ["1K", "2K", "4K"]
seed = 0  # @param {type:"integer"}
output_format = "png"  # @param ["png", "jpeg", "webp"]

images = generate_image(
    model="black-forest-labs/flux.2-pro",
    prompt=prompt,
    aspect_ratio=aspect_ratio,
    resolution=resolution,
    seed=seed,
    output_format=output_format,
)
show_and_save(images, output_format, prefix="flux2_pro")

# Edit

FLUX.2 Pro also edits existing images: pass an input image via `input_references` together
with an edit prompt. Run the cell, click the upload button (or leave the path empty and use
a Drive path), and the image is converted to a base64 data URL automatically.

In [ ]:
# @title FLUX.2 Pro — Image Edit

# @markdown - Click the upload button below, **or**
# @markdown - Leave the upload empty and paste a Drive path here:
image_path = ""  # @param {type:"string"}
prompt = "Make it a rainy neon-lit night scene with reflective streets"  # @param {type:"string"}

if image_path:
    reference = to_data_url(image_path)
else:
    uploaded = files.upload()
    reference = to_data_url(next(iter(uploaded)))

images = generate_image(
    model="black-forest-labs/flux.2-pro",
    prompt=prompt,
    input_references=[{"type": "image", "image_url": reference}],
)
show_and_save(images, prefix="flux2_pro_edit")

# Budget

FLUX.2 Klein 4B is the fastest and most cost-effective model in the FLUX.2 family —
ideal for drafts and high-volume runs while keeping solid image quality.

See <https://openrouter.ai/black-forest-labs>

In [ ]:
# @title FLUX.2 Klein 4B — Budget Text to Image

prompt = "op art cat illusion red blue chromostereopsis maximum saturation"  # @param {type:"string"}
aspect_ratio = "1:1"  # @param ["1:1", "16:9", "9:16", "4:3", "3:2", "21:9"]
resolution = "1K"  # @param ["1K", "2K", "4K"]
seed = 0  # @param {type:"integer"}
output_format = "png"  # @param ["png", "jpeg", "webp"]

images = generate_image(
    model="black-forest-labs/flux.2-klein-4b",
    prompt=prompt,
    aspect_ratio=aspect_ratio,
    resolution=resolution,
    seed=seed,
    output_format=output_format,
)
show_and_save(images, output_format, prefix="flux2_klein")